In [2]:
import pickle
with open('../data/processed_chunks.pkl', 'rb') as f:
    processed_chunks = pickle.load(f)

In [5]:
import os
import pickle
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Load your chunks from Notebook 1
try:
    with open('../data/processed_chunks.pkl', 'rb') as f:
        processed_chunks = pickle.load(f)
    print(f"Successfully loaded {len(processed_chunks)} chunks.")
except FileNotFoundError:
    print("Error: processed_chunks.pkl not found!")

# 2. Initialize HuggingFace Embeddings (Local Model)
print("--- Loading local HuggingFace Model (all-MiniLM-L6-v2) ---")
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'} 
)

# 3. Create the Database
persist_directory = "../db"
print("--- Creating Vector Database (This may take a moment on CPU) ---")

vector_db = Chroma.from_documents(
    documents=processed_chunks, 
    embedding=embeddings,
    persist_directory=persist_directory
)

# 4. Verify Dimension (Scientific Proof for Supervisor)
# MiniLM-L6-v2 always produces 384-dimensional vectors
sample_vector = embeddings.embed_query("KazNU Academic Policy")
print(f"Success! Embedding dimension size: {len(sample_vector)}")

# 5. TEST SEARCH
query = "What are the rules for academic mobility?"
docs = vector_db.similarity_search(query, k=2)

print("\n--- Semantic Search Results ---")
for i, doc in enumerate(docs):
    print(f"[{doc.metadata['source_type'].upper()}] Result {i+1}:")
    print(f"{doc.page_content[:200]}...\n")

Successfully loaded 200 chunks.
--- Loading local HuggingFace Model (all-MiniLM-L6-v2) ---


d:\RAG_StudentSupport_Project\agentic-rag-edu-4th\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1534.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Creating Vector Database (This may take a moment on CPU) ---
Success! Embedding dimension size: 384

--- Semantic Search Results ---
[POLICY] Result 1:
department, as well as instructors of pedagogy and psychology (depending on the educational 
program). The results of research and industrial internships are presented in the form of a written 
report...

[POLICY] Result 2:
13.2 The direction for participation in academic mobility, financed from the state budget, is 
carried out in accordance with the Rules for the direction for studying abroad, including within the 
fra...

